<div align="center">

# **Forward and Backward Propagation**

**Laboratory 3: Deep Learning Fundamentals**

</div>


This laboratory extends the forward-pass computation from Laboratory 2 by adding backward propagation and parameter updating. The goal is to show how a neural network uses its prediction error to determine how its weights and biases should change.

## **Objectives**

By the end of this laboratory, the student should be able to:

- perform forward propagation using the given network parameters;
- compute the output and hidden-layer errors;
- determine the gradients of the weights and biases; and
- update the network parameters using the learning rate.

## **Laboratory Problem**

> **Instruction:** Perform a forward and backward propagation in Python using the inputs from Laboratory Task 2.

Given:

$$
x =
\begin{bmatrix}
1 & 0 & 1
\end{bmatrix},
\qquad
y = [1]
$$

Use the **ReLU activation function** and:

$$
\text{learning rate} = 0.001
$$

The initial weights and biases are taken from Laboratory 2.

## **Concept Review**

### Backward Propagation

Backward propagation is the process a neural network uses to learn from its prediction error. After the forward pass produces a prediction $\hat{y}$, it is compared with the actual target $y$ using:

$$
\delta_o = \hat{y} - y
$$

The error is then propagated backward through the network to determine how the weights and biases contributed to the result. For the hidden layer, the error is computed as:

$$
\delta_h = (\delta_o W_o^T)\cdot f'(Z_h)
$$

For ReLU, the derivative is:

$$
f'(Z)=
\begin{cases}
1, & Z>0 \\
0, & Z\leq0
\end{cases}
$$

This means that only neurons with positive pre-activation values can pass a gradient backward during that step. The gradient indicates how much and in what direction a parameter should change to reduce the error. The weight gradients are:

$$
\frac{\partial E}{\partial W_o}=A_h^T\delta_o
\qquad
\frac{\partial E}{\partial W_h}=X^T\delta_h
$$

The parameters are then updated using the learning rate $\alpha$:

$$
W_{\text{new}} = W_{\text{old}} - \alpha\frac{\partial E}{\partial W}
$$

$$
b_{\text{new}} = b_{\text{old}} - \alpha\frac{\partial E}{\partial b}
$$

Conceptually, backpropagation follows the cycle:

$$
\text{Prediction} \rightarrow \text{Error} \rightarrow \text{Gradients} \rightarrow \text{Parameter Update}
$$

This allows the network to gradually adjust its parameters so that future predictions move closer to the target.

## **Implementation**

#### **Define the Inputs and Network Parameters**
The inputs, target output, weights, and biases are taken from Laboratory 2. A learning rate of $0.001$ is used to control the size of each parameter update.

In [1]:
import numpy as np

In [2]:
# Input and target
x = np.array([1.0, 0.0, 1.0])
y = np.array([1.0])

# Hidden-layer weights
W_hidden = np.array([
    [0.2, -0.3],
    [0.4,  0.1],
    [-0.5, 0.2]
])

# Output-layer weights
W_output = np.array([
    [-0.3],
    [-0.2]
])

# Biases
b_hidden = np.array([-0.4, 0.2])
b_output = np.array([0.1])

# Learning rate
lr = 0.001

#### **Define ReLU and Its Derivative**


In [3]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

#### **Forward Propagation**

Using the same network from Laboratory 2:

$$
Z_1 = (0.2)(1) + (0.4)(0) + (-0.5)(1) + (-0.4) = -0.7
$$

$$
Z_2 = (-0.3)(1) + (0.1)(0) + (0.2)(1) + (0.2) = 0.1
$$

Applying ReLU:

$$
H_1 = \max(0,-0.7)=0,
\qquad
H_2 = \max(0,0.1)=0.1
$$

The output neuron is:

$$
\hat{y} = (-0.3)(0)+(-0.2)(0.1)+(0.1)=0.08
$$

The output error is:

$$
\delta_o=\hat{y}-y=0.08-1=-0.92
$$

In [4]:
# Forward propagation
z_hidden = x @ W_hidden + b_hidden
a_hidden = relu(z_hidden)

z_output = a_hidden @ W_output + b_output
y_hat = relu(z_output)

delta_output = y_hat - y

print("Hidden weighted sums:", z_hidden)
print("Hidden activations:", a_hidden)
print("Predicted output:", y_hat)
print("Output error:", delta_output)

Hidden weighted sums: [-0.7  0.1]
Hidden activations: [0.  0.1]
Predicted output: [0.08]
Output error: [-0.92]


#### **Backward Propagation**

The hidden-layer error is:

$$
\delta_h=(\delta_oW_o^T)\cdot f'(Z_h)
$$

Since:

$$
W_o^T=
\begin{bmatrix}
-0.3 & -0.2
\end{bmatrix}
\qquad
f'(Z_h)=
\begin{bmatrix}
0 & 1
\end{bmatrix}
$$

then:

$$
\delta_h=
(-0.92)
\begin{bmatrix}
-0.3 & -0.2
\end{bmatrix}
\cdot
\begin{bmatrix}
0 & 1
\end{bmatrix}
=
\begin{bmatrix}
0 & 0.184
\end{bmatrix}
$$

In [5]:
# Backward propagation
delta_hidden = (delta_output @ W_output.T) * relu_derivative(z_hidden)

#### **Gradient Computation**

For the output-layer weights:

$$
\frac{\partial E}{\partial W_o}
=
A_h^T\delta_o
=
\begin{bmatrix}
0 \\
0.1
\end{bmatrix}
(-0.92)
=
\begin{bmatrix}
0 \\
-0.092
\end{bmatrix}
$$

For the hidden-layer weights:

$$
\frac{\partial E}{\partial W_h}
=
X^T\delta_h
=
\begin{bmatrix}
1 \\
0 \\
1
\end{bmatrix}
\begin{bmatrix}
0 & 0.184
\end{bmatrix}
=
\begin{bmatrix}
0 & 0.184 \\
0 & 0 \\
0 & 0.184
\end{bmatrix}
$$

In [6]:
# Gradients
grad_W_output = np.outer(a_hidden, delta_output)
grad_b_output = delta_output.copy()

grad_W_hidden = np.outer(x, delta_hidden)
grad_b_hidden = delta_hidden.copy()

print("Hidden-layer error:", delta_hidden)
print("\nOutput-layer weight gradients:\n", grad_W_output)
print("\nHidden-layer weight gradients:\n", grad_W_hidden)

Hidden-layer error: [0.    0.184]

Output-layer weight gradients:
 [[-0.   ]
 [-0.092]]

Hidden-layer weight gradients:
 [[0.    0.184]
 [0.    0.   ]
 [0.    0.184]]


#### **Parameter Update**

Using:

$$
W_{\text{new}}
=
W_{\text{old}}
-
\alpha\frac{\partial E}{\partial W}
$$

with:

$$
\alpha=0.001
$$

the updated output-layer weights are:

$$
W_o=
\begin{bmatrix}
-0.3 \\
-0.2
\end{bmatrix}
-
0.001
\begin{bmatrix}
0 \\
-0.092
\end{bmatrix}
=
\begin{bmatrix}
-0.3 \\
-0.199908
\end{bmatrix}
$$

The updated hidden-layer weights are:

$$
W_h=
\begin{bmatrix}
0.2 & -0.300184 \\
0.4 & 0.1 \\
-0.5 & 0.199816
\end{bmatrix}
$$

The updated biases are:

$$
b_h=
\begin{bmatrix}
-0.4 & 0.199816
\end{bmatrix}
\qquad
b_o=0.10092
$$

In [7]:
# Update parameters
W_output -= lr * grad_W_output
b_output -= lr * grad_b_output

W_hidden -= lr * grad_W_hidden
b_hidden -= lr * grad_b_hidden

print("Updated hidden weights:\n", W_hidden)
print("Updated hidden biases:", b_hidden)

print("\nUpdated output weights:\n", W_output)
print("Updated output bias:", b_output)

Updated hidden weights:
 [[ 0.2      -0.300184]
 [ 0.4       0.1     ]
 [-0.5       0.199816]]
Updated hidden biases: [-0.4       0.199816]

Updated output weights:
 [[-0.3     ]
 [-0.199908]]
Updated output bias: [0.10092]


#### **Verification After the Update**

After updating the weights and biases, another forward pass is performed to check whether the prediction moves closer to the target.

$$
Z_2=(-0.300184)(1)+(0.1)(0)+(0.199816)(1)+(0.199816)=0.099448
$$

$$
H_2=\max(0,0.099448)=0.099448
$$

The updated output is:

$$
\hat{y}_{new}=(-0.3)(0)+(-0.199908)(0.099448)+0.10092
$$

$$
\hat{y}_{new}\approx0.08104
$$

Thus, the updated error is:

$$
\delta_{new}=0.08104-1=\boxed{-0.91896}
$$

Since $|-0.91896|<|-0.92|$, the prediction moved slightly closer to the target.

In [8]:
# Verify updated prediction and error
new_z_hidden = x @ W_hidden + b_hidden
new_a_hidden = relu(new_z_hidden)

new_z_output = new_a_hidden @ W_output + b_output
new_y_hat = relu(new_z_output)

new_error = new_y_hat - y

print("Updated prediction:", new_y_hat)
print("Updated error:", new_error)

Updated prediction: [0.08103955]
Updated error: [-0.91896045]


## **Results and Discussion**

The initial forward pass produced a prediction of $\hat{y}=0.08$ with an error of $\delta_o=-0.92$. During backpropagation, only the second hidden neuron received a nonzero hidden-layer error of $0.184$, so its connected weights and bias were updated while the inactive first neuron remained unchanged. After applying the learning rate of $0.001$, the updated network produced a new prediction of approximately $\hat{y}=0.08104$ and a new error of $\delta=-0.91896$. Since the magnitude of the error decreased from $0.92$ to $0.91896$, the prediction moved slightly closer to the target value of $1$. This shows that the parameter update improved the network, although the change was small because the learning rate was intentionally low.

## **Conclusion** 

This laboratory demonstrated how backpropagation allows a neural network to improve by adjusting its parameters based on the error from the forward pass. The second hidden neuron received a nonzero gradient and its connected parameters were updated, while the inactive first neuron remained unchanged because of ReLU. After one update, the prediction increased from $0.08$ to approximately $0.08104$, while the error magnitude decreased from $0.92$ to $0.91896$. Although the improvement was small, it confirms that the update moved the network slightly closer to the target and illustrates how repeated training steps gradually improve model performance.

This laboratory helped me understand how a neural network actually learns from its mistakes. I saw that backpropagation does not change all parameters equally because only the neurons and connections that contribute to the error receive meaningful updates. The small improvement in the prediction after one update also showed me that learning happens gradually, especially when using a small learning rate. This made the relationship between error, gradients, parameter updates, and model improvement much clearer to me.